In [3]:
import os, numpy as np, pandas as pd, scanpy as sc
from pathlib import Path
from scipy import sparse

In [76]:
IN_H5AD = "MAIT_NKT_gdT_paperlabels_exact.h5ad"
TF_LIST = "tf_list_used.txt"       
TF_COMMUNITY_TSV = "tf_multiplex_communities_step_80pct.csv"       
OUT_H5AD = "gCRL/data/real/Lee/lee_full.h5ad"
os.makedirs(Path(OUT_H5AD).parent, exist_ok=True)

In [49]:
ad = sc.read_h5ad(IN_H5AD)
ad

AnnData object with n_obs × n_vars = 8253 × 25970
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'nGene', 'nUMI', 'use', 'final_celltype', 'major_type', 'paper_subtype', 'n_counts', 'n_genes', 'author_cluster', 'author_lineage', 'paper_cluster', 'paper_lineage'
    var: 'gene_symbols', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'gene_symbol'
    uns: 'log1p', 'paper_subtype_colors', 'umap_provenance'
    obsm: 'X_PCA', 'X_TSNE', 'X_umap'
    layers: 'counts'

In [50]:
def ensure_log1p(ad):
    X = ad.X.A if sparse.issparse(ad.X) else ad.X
    if np.max(X) > 50 and "counts" in ad.layers:
        ad = ad.copy()
        ad.X = ad.layers["counts"].copy()
        sc.pp.normalize_total(ad, target_sum=1e4)
        sc.pp.log1p(ad)
    return ad

In [51]:
ad = ensure_log1p(ad)

In [52]:
map_nkt = {"N1":"NKTp","N2":"NKT1","N3":"NKT2","N4":"NKT2","N5":"NKT2","N6":"NKT2","N7":"NKT17"}
map_mait_sub = {"M1":"MAIT0","M2":"MAIT2","M3":"MAIT1/17i","M4":"MAIT1i","M5":"MAIT1","M6":"MAIT17i","M7":"MAIT17i","M8":"MAIT17"}
map_mait_stage = {"M1":"S1","M2":"S2","M3":"S2","M4":"S2","M5":"S3","M6":"S3","M7":"S3","M8":"S3"}
map_gdt = {"G1":"Tγδp","G2":"Tγδ1/17i","G3":"Tγδ1/17i","G4":"Tγδ17i","G5":"Tγδ17i",
           "G6-1":"Tγδ17","G6-2":"Tγδ17","G7-1":"Tγδ1i","G7-2":"Tγδ1"}

def sub(cl):  
    cl = str(cl)
    return map_nkt.get(cl, map_mait_sub.get(cl, map_gdt.get(cl, "NA")))

def stage(cl):
    return map_mait_stage.get(str(cl), "NA")

if "paper_subtype_strict" not in ad.obs:
    ad.obs["paper_subtype_strict"] = ad.obs["paper_cluster"].astype(str).map(sub)

if "paper_stage" not in ad.obs:
    if "paper_cluster" in ad.obs:
        ad.obs["paper_stage"] = ad.obs["paper_cluster"].astype(str).map(stage)
    else:
        ad.obs["paper_stage"] = "NA"

for c in ad.obs.columns:
    ad.obs[c] = ad.obs[c].astype(str)

In [53]:
lin = ad.obs["paper_lineage"].astype(str)
sub = ad.obs["paper_subtype_strict"].astype(str)
stage = ad.obs.get("paper_stage", pd.Series(index=ad.obs.index, dtype=object)).astype(str)

In [54]:
is_MAIT = lin.eq("MAIT").values
MAIT_ctrl = (is_MAIT & (sub.isin(["S1","S2","MAIT0","MAIT2","MAIT1/17i","MAIT1i"]) | stage.isin(["S1","S2"])))
MAIT_1    = (is_MAIT & sub.eq("MAIT1"))
MAIT_17   = (is_MAIT & sub.eq("MAIT17"))

In [55]:
is_NKT = lin.eq("NKT").values
NKT_ctrl = (is_NKT & sub.eq("NKTp"))
NKT_1    = (is_NKT & sub.eq("NKT1"))
NKT_17   = (is_NKT & sub.eq("NKT17"))

In [56]:
is_GDT = lin.isin(["γδT","gdT"]).values
GDT_ctrl = (is_GDT & sub.isin(["Tγδp","gdTp","Tγδ1/17i","gdT1i"])) 
GDT_1    = (is_GDT & sub.isin(["Tγδ1","gdT1"]))
GDT_17   = (is_GDT & sub.isin(["Tγδ17","gdT17"]))

In [57]:
use_mask = (MAIT_ctrl | MAIT_1 | MAIT_17 | NKT_ctrl | NKT_1 | NKT_17 | GDT_ctrl | GDT_1 | GDT_17).values
ad = ad[use_mask].copy()

In [58]:
# cell_type
ad.obs["cell_type"] = ad.obs["paper_lineage"].astype(str)

In [59]:
# intervention: controls - "unperturbed"; Tbx21 finals - "Tbx21"; Rorc finals - "Rorc"
interv = np.full(ad.n_obs, "unperturbed", dtype=object)
interv[MAIT_1.loc[ad.obs_names].values | NKT_1.loc[ad.obs_names].values | GDT_1.loc[ad.obs_names].values] = "Tbx21"
interv[MAIT_17.loc[ad.obs_names].values | NKT_17.loc[ad.obs_names].values | GDT_17.loc[ad.obs_names].values] = "Rorc"
ad.obs["intervention"] = pd.Categorical(interv, categories=["unperturbed","Tbx21","Rorc"])

In [60]:
rng = np.random.RandomState(42)

def split_train_test(idx):
    if idx.sum() == 0: 
        return pd.Series(False, index=ad.obs_names)
    ids = np.where(idx)[0]
    rng.shuffle(ids)
    k = int(0.7 * len(ids))
    keep = np.zeros(ad.n_obs, dtype=bool)
    keep[ids[:k]] = True
    return pd.Series(keep, index=ad.obs_names)

In [61]:
train = pd.Series(False, index=ad.obs_names)
train = train | pd.Series((ad.obs["intervention"]=="unperturbed").values, index=ad.obs_names)

In [62]:
for ct in ["MAIT","NKT","γδT"]:
    for iv in ["Tbx21","Rorc"]:
        mask = (ad.obs["cell_type"]==ct) & (ad.obs["intervention"]==iv)
        train = train | split_train_test(mask.values)

ad.obs["set"] = pd.Categorical(np.where(train, "training", "test"), categories=["training","test"])

In [63]:
# var.kind and var.community
tf_set = {g.strip() for g in Path(TF_LIST).read_text().splitlines() if g.strip()}
ad.var["kind"] = np.where(ad.var_names.isin(tf_set), "TF", "TG")

In [64]:
#community from multiplex clusters (only for TFs), others = -1
comm_df = pd.read_csv(
    TF_COMMUNITY_TSV,
    sep=None, engine="python")

In [65]:
# normalize column names
cols_lower = {c.lower(): c for c in comm_df.columns}
# find TF column
tf_col = None
for key in ["tf", "gene", "symbol", "gene_name"]:
    if key in cols_lower:
        tf_col = cols_lower[key]; break
if tf_col is None:
    raise ValueError(f"Could not find a TF column in {TF_COMMUNITY_TSV} "
                     f"Columns found: {list(comm_df.columns)}")

In [66]:
comm_col = None
for key in ["community", "module", "cluster", "leiden", "comm"]:
    if key in cols_lower:
        comm_col = cols_lower[key]; break
if comm_col is None:
    raise ValueError(f"Could not find a community column in {TF_COMMUNITY_TSV}"
                     f"Columns found: {list(comm_df.columns)}")

In [67]:
cdf = comm_df[[tf_col, comm_col]].copy()
cdf[tf_col] = cdf[tf_col].astype(str).str.strip()
cdf[comm_col] = pd.to_numeric(cdf[comm_col], errors="coerce")

In [68]:
cdf = (cdf.dropna(subset=[tf_col])
          .sort_values(by=[tf_col, comm_col], na_position="last")
          .drop_duplicates(subset=[tf_col], keep="first"))

tf2comm = pd.Series(cdf[comm_col].values, index=cdf[tf_col]).to_dict()
ad.var["community"] = -1

In [69]:
mask_tf = (ad.var["kind"] == "TF")
tf_names_in_ad = ad.var.index[mask_tf]

In [70]:
mapped_vals = []
miss_tf = []
for g in tf_names_in_ad:
    v = tf2comm.get(str(g), np.nan)
    if pd.isna(v):
        mapped_vals.append(-1)
        miss_tf.append(g)
    else:
        mapped_vals.append(int(v))
ad.var.loc[mask_tf, "community"] = mapped_vals


In [71]:
n_tf_total = int(mask_tf.sum())
n_tf_mapped = int((ad.var.loc[mask_tf, "community"] != -1).sum())
print(f"[community] TF mapped: {n_tf_mapped}/{n_tf_total} "
      f"({n_tf_mapped/n_tf_total:.1%})")
if miss_tf:
    print("[community] Example TF without community (first 10):", list(miss_tf)[:10])

[community] TF mapped: 163/163 (100.0%)


In [72]:
def cnt(mask):
    return int(mask.sum())


In [73]:
MAIT_ctrl_n = cnt(MAIT_ctrl.loc[ad.obs_names])
MAIT1_n     = cnt(MAIT_1.loc[ad.obs_names])
MAIT17_n    = cnt(MAIT_17.loc[ad.obs_names])

GDT_ctrl_n  = cnt(GDT_ctrl.loc[ad.obs_names])
GDT1_n      = cnt(GDT_1.loc[ad.obs_names])
GDT17_n     = cnt(GDT_17.loc[ad.obs_names])

NKT_ctrl_n  = cnt(NKT_ctrl.loc[ad.obs_names])
NKT1_n      = cnt(NKT_1.loc[ad.obs_names])
NKT17_n     = cnt(NKT_17.loc[ad.obs_names])

In [74]:
print("Expected CVAE-like counts:")
print("MAIT ctrl/1/17:", MAIT_ctrl_n, MAIT1_n, MAIT17_n, "(should be 1331 / 148 / 464)")
print("γδT  ctrl/1/17:", GDT_ctrl_n,  GDT1_n,  GDT17_n,  "(943 / 76 / 230)")
print("NKT  ctrl/1/17:", NKT_ctrl_n,  NKT1_n,  NKT17_n,  "(225 / 210 / 283)")

print("\nCounts by cell_type x intervention:")
print(pd.crosstab(ad.obs["cell_type"], ad.obs["intervention"]))

print("\nSplit check (counts):")
print(pd.crosstab(ad.obs["set"], ad.obs["intervention"]))

print("\nControl presence in TRAIN by cell_type:")
print(pd.crosstab(ad.obs["cell_type"], (ad.obs["intervention"]=="unperturbed") & (ad.obs["set"]=="training")))

print("\nvar.kind counts:", dict(ad.var["kind"].value_counts()))
print("community unique (first 10):", sorted(ad.var["community"].unique())[:10])

Expected CVAE-like counts:
MAIT ctrl/1/17: 1331 148 464 (should be 1331 / 148 / 464)
γδT  ctrl/1/17: 943 76 230 (943 / 76 / 230)
NKT  ctrl/1/17: 225 210 283 (225 / 210 / 283)

Counts by cell_type x intervention:
intervention  unperturbed  Tbx21  Rorc
cell_type                             
MAIT                 1331    148   464
NKT                   225    210   283
γδT                   943     76   230

Split check (counts):
intervention  unperturbed  Tbx21  Rorc
set                                   
training             2499    303   683
test                    0    131   294

Control presence in TRAIN by cell_type:
col_0      False  True 
cell_type              
MAIT         612   1331
NKT          493    225
γδT          306    943

var.kind counts: {'TG': 25807, 'TF': 163}
community unique (first 10): [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8]


In [75]:
ad

AnnData object with n_obs × n_vars = 3910 × 25970
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'nGene', 'nUMI', 'use', 'final_celltype', 'major_type', 'paper_subtype', 'n_counts', 'n_genes', 'author_cluster', 'author_lineage', 'paper_cluster', 'paper_lineage', 'paper_subtype_strict', 'paper_stage', 'cell_type', 'intervention', 'set'
    var: 'gene_symbols', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'gene_symbol', 'kind', 'community'
    uns: 'log1p', 'paper_subtype_colors', 'umap_provenance'
    obsm: 'X_PCA', 'X_TSNE', 'X_umap'
    layers: 'counts'

In [77]:
ad.write_h5ad(OUT_H5AD, compression="gzip")
print("\nSaved:", OUT_H5AD)


Saved: gCRL/data/real/Lee/lee_full.h5ad


In [93]:
#only necessary columns
obs_keep = ["cell_type", "intervention", "set"]
var_keep = ["kind", "community"]

ad.obs = ad.obs[obs_keep].copy()
ad.var = ad.var[var_keep].copy()

ad.obs["cell_type"]    = ad.obs["cell_type"].astype("category")
ad.obs["intervention"] = ad.obs["intervention"].astype("category")
ad.obs["set"]          = ad.obs["set"].astype("category")

ad.var["kind"]      = ad.var["kind"].astype("category")
ad.var["community"] = ad.var["community"].astype(int)


ad.obsm.clear(); ad.obsp.clear(); ad.uns.clear()
if sparse.issparse(ad.X):
    ad.X = ad.X.astype(np.float32)
else:
    ad.X = ad.X.astype(np.float32, copy=False)



In [94]:
ad

AnnData object with n_obs × n_vars = 3910 × 25970
    obs: 'cell_type', 'intervention', 'set'
    var: 'kind', 'community'
    varm: 'PCs'
    layers: 'counts'

In [95]:
ct_order  = [x for x in ["MAIT","NKT","γδT"] if x in ad.obs["cell_type"].cat.categories]
int_order = [x for x in ["unperturbed","Tbx21","Rorc"] if x in ad.obs["intervention"].cat.categories]
set_order = [x for x in ["training","test"] if x in ad.obs["set"].cat.categories]

ct_colors = {
    "MAIT": "#1f77b4", 
    "NKT":  "#2ca02c", 
    "γδT":  "#d62728", 
}
int_colors = {
    "unperturbed": "#7f7f7f", 
    "Tbx21":       "#1f77b4",  
    "Rorc":        "#ff7f0e",  
}

ad.uns["cell_type_colors"]    = [ct_colors[c]  for c in ad.obs["cell_type"].cat.categories]
ad.uns["intervention_colors"] = [int_colors[i] for i in ad.obs["intervention"].cat.categories]


ad.obs["ct_int"] = (ad.obs["cell_type"].astype(str) + "|" +
                    ad.obs["intervention"].astype(str)).astype("category")

if MAKE_UMAP:

    sc.tl.pca(ad, n_comps=25, svd_solver="arpack")
    sc.pp.neighbors(ad, n_neighbors=40, metric="cosine")
    sc.tl.umap(ad, min_dist=0.45)

ad.uns["gcrl_meta"] = {
    "note": "Lee dataset in gCRL format; slim + viz conveniences",
    "counts_by_cell_type": pd.crosstab(ad.obs["cell_type"], ad.obs["intervention"]).to_dict(),
    "split_counts": pd.crosstab(ad.obs["set"], ad.obs["intervention"]).to_dict(),
    "var_kind_counts": ad.var["kind"].value_counts().to_dict(),
}



In [96]:
ad

AnnData object with n_obs × n_vars = 3910 × 25970
    obs: 'cell_type', 'intervention', 'set', 'ct_int'
    var: 'kind', 'community'
    uns: 'cell_type_colors', 'intervention_colors', 'pca', 'neighbors', 'umap', 'gcrl_meta'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [98]:
OUT_VIZ= "gCRL/data/real/Lee/lee_gcrl_viz.h5ad"
Path(OUT_VIZ).parent.mkdir(parents=True, exist_ok=True)
ad.write(OUT_VIZ, compression="gzip")
print("Viz-ready saved:", OUT_VIZ)


Viz-ready saved: gCRL/data/real/Lee/lee.h5ad


In [4]:
IN_H5AD = "gCRL/data/real/Lee/lee_gcrl_viz.h5ad"
ad = sc.read_h5ad(IN_H5AD)
ad

AnnData object with n_obs × n_vars = 3910 × 25970
    obs: 'cell_type', 'intervention', 'set', 'ct_int'
    var: 'kind', 'community'
    uns: 'cell_type_colors', 'gcrl_meta', 'intervention_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [5]:
OUT_H5AD = "gCRL/data/real/Lee/lee.h5ad"

FNKT_TXT   = "exports_Fbest_withTF/NKT/var_names.txt"          # 2781 genes
FMAIT_TXT  = "exports_Fbest_withTF/MAIT/var_names.txt"  # 2782 genes (H100 ∪ {Ccr7})
FGDT_TXT   = "exports_Fbest_withTF/gdT/var_names.txt"           # 3255 genes


genes = list(ad.var_names)

def load_gene_list(path):
    s = Path(path).read_text().splitlines()
    s = [g.strip() for g in s if g.strip()]
    return set(s)

F_NKT  = load_gene_list(FNKT_TXT)
F_MAIT = load_gene_list(FMAIT_TXT)
F_gdT  = load_gene_list(FGDT_TXT)

print(f"|F_NKT|={len(F_NKT)}  |F_MAIT|={len(F_MAIT)}  |F_gdT|={len(F_gdT)}")
print("Ccr7 in MAIT panel:", "Ccr7" in F_MAIT)


|F_NKT|=2781  |F_MAIT|=2782  |F_gdT|=3255
Ccr7 in MAIT panel: True


In [6]:
ad.var["in_panel_NKT_H100"]       = ad.var_names.isin(F_NKT).astype("bool")
ad.var["in_panel_MAIT_H100_Ccr7"] = ad.var_names.isin(F_MAIT).astype("bool")
ad.var["in_panel_gdT_boost2"]     = ad.var_names.isin(F_gdT).astype("bool")

ad.var["in_panel_union"] = (
    ad.var["in_panel_NKT_H100"] |
    ad.var["in_panel_MAIT_H100_Ccr7"] |
    ad.var["in_panel_gdT_boost2"]
).astype("bool")

def which_panels(row):
    tags = []
    if row["in_panel_NKT_H100"]:       tags.append("NKT_H100")
    if row["in_panel_MAIT_H100_Ccr7"]: tags.append("MAIT_H100+Ccr7")
    if row["in_panel_gdT_boost2"]:     tags.append("gdT_boost2")
    return "+".join(tags) if tags else ""

ad.var["panel_membership"] = ad.var.apply(which_panels, axis=1).astype("string")

ad.uns.setdefault("panels_meta", {})
ad.uns["panels_meta"]["NKT_H100"]       = {"n_genes": len(F_NKT)}
ad.uns["panels_meta"]["MAIT_H100+Ccr7"] = {"n_genes": len(F_MAIT), "note": "H100 union plus marker Ccr7"}
ad.uns["panels_meta"]["gdT_boost2"]     = {"n_genes": len(F_gdT), "note": "boosted set (HVGs + lineage markers)"}


In [7]:
def count_in(flag): 
    return int(ad.var[flag].sum())
print("\nCounts in ad.var (present in this AnnData):")
print("NKT_H100:", count_in("in_panel_NKT_H100"))
print("MAIT_H100+Ccr7:", count_in("in_panel_MAIT_H100_Ccr7"))
print("gdT_boost2:", count_in("in_panel_gdT_boost2"))
print("panel UNION:", int(ad.var["in_panel_union"].sum()))

if "Ccr7" in ad.var_names:
    print("Ccr7 flags — MAIT_H100+Ccr7:", bool(ad.var.loc["Ccr7", "in_panel_MAIT_H100_Ccr7"]))



Counts in ad.var (present in this AnnData):
  NKT_H100        : 2781
  MAIT_H100+Ccr7  : 2782
  gdT_boost2      : 3255
  panel UNION     : 3256
Ccr7 flags — MAIT_H100+Ccr7: True


In [8]:
ad

AnnData object with n_obs × n_vars = 3910 × 25970
    obs: 'cell_type', 'intervention', 'set', 'ct_int'
    var: 'kind', 'community', 'in_panel_NKT_H100', 'in_panel_MAIT_H100_Ccr7', 'in_panel_gdT_boost2', 'in_panel_union', 'panel_membership'
    uns: 'cell_type_colors', 'gcrl_meta', 'intervention_colors', 'neighbors', 'pca', 'umap', 'panels_meta'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [ ]:
ad.write_h5ad(OUT_H5AD, compression="gzip")
print("Saved:", OUT_H5AD)